In [22]:
import pandas as pd
import numpy as np
import ast
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [23]:
df = pd.read_csv('system_rekomendacji_final_clean.csv')

In [26]:
print(f"Załadowano {len(df)} filmów.")
df.head()
df.info()
df.describe()

Załadowano 4803 filmów.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    4803 non-null   int64  
 1   title                 4803 non-null   object 
 2   genres                4803 non-null   object 
 3   keywords              4803 non-null   object 
 4   cast                  4803 non-null   object 
 5   director              4803 non-null   object 
 6   overview              4800 non-null   object 
 7   popularity            4803 non-null   float64
 8   vote_average          4803 non-null   float64
 9   vote_count            4803 non-null   int64  
 10  release_date          4803 non-null   object 
 11  budget                3766 non-null   float64
 12  revenue               3376 non-null   float64
 13  runtime               4803 non-null   float64
 14  original_language     4803 non-null   object 
 1

,id,popularity,vote_average,vote_count,budget,revenue,runtime,release_year,return_on_investment,weighted_score,soup_length
count,4803.000000,4803.000000,4803.000000,4803.000000,3.766000e+03,3.376000e+03,4803.000000,4803.000000,4.803000e+03,4803.000000,4803.000000
mean,57165.484281,21.492301,6.092172,690.217989,3.704284e+07,1.170314e+08,107.632521,2002.469498,1.985708e+03,6.180409,175.788049
std,88694.614033,31.816650,1.194612,1234.585891,4.264651e+07,1.834831e+08,20.671809,12.413166,1.234915e+05,0.272499,71.907245
min,5.000000,0.000000,0.000000,0.000000,1.000000e+00,5.000000e+00,14.000000,1916.000000,-1.000000e+00,5.155730,3.000000
25%,9014.500000,4.668070,5.600000,54.000000,8.000000e+06,1.535290e+07,94.000000,1999.000000,-1.627699e-01,6.072610,127.000000
50%,14629.000000,12.921594,6.200000,235.000000,2.300000e+07,5.175184e+07,104.000000,2005.000000,4.659315e-02,6.095519,163.000000
75%,58610.500000,28.313505,6.800000,737.000000,5.000000e+07,1.401651e+08,117.500000,2011.000000,2.157066e+00,6.193293,212.000000
max,459488.000000,875.581305,10.000000,13752.000000,3.800000e+08,2.787965e+09,338.000000,2017.000000,8.499999e+06,8.059258,1259.000000


In [27]:
# 1. Definicja sukcesu (Top 25% filmów pod względem weighted_score)
threshold = df['weighted_score'].quantile(0.75)
df['is_success'] = (df['weighted_score'] > threshold).astype(int)

In [28]:
# 2. Przygotowanie wektoryzatora TF-IDF
# Zamieniamy tekst na liczby, ignorując popularne angielskie słowa
tfidf = TfidfVectorizer(stop_words='english', max_features=2000)
X = tfidf.fit_transform(df['metadata_soup'].fillna(''))
y = df['is_success']

In [29]:
# 3. Trening modelu logistycznego
model = LogisticRegression()
model.fit(X, y)

print(f"Model został wytrenowany. Próg sukcesu: {threshold:.2f} pkt.")

Model został wytrenowany. Próg sukcesu: 6.19 pkt.


In [30]:
def predict_success(soup):
    """Oblicza prawdopodobieństwo sukcesu dla podanej zupy słów."""
    # Czyszczenie wejścia (małe litery)
    soup = soup.lower().strip()
    # Transformacja tekstu na format zrozumiały dla modelu
    vectorized_input = tfidf.transform([soup])
    # Pobranie prawdopodobieństwa dla klasy 1 (sukces)
    probability = model.predict_proba(vectorized_input)[0][1]
    return round(probability * 100, 2)

In [31]:
def get_top_items(column, top_n=50):
    def safe_parse(val):
        try: return ast.literal_eval(val) if isinstance(val, str) else []
        except: return []
    all_items = [item for sublist in df[column].apply(safe_parse) for item in sublist]
    return [i[0] for i in Counter(all_items).most_common(top_n)]

print("🎭 GATUNKI:", sorted(list(set([g for sublist in df['genres'].apply(ast.literal_eval) for g in sublist]))))
print("\n🔑 TEMATY (Keywords):", get_top_items('keywords'))
print("\n🌟 AKTORZY:", get_top_items('cast'))
print("\n🎬 REŻYSERZY:", df['director'].value_counts().head(50).index.tolist())

🎭 GATUNKI: ['action', 'adventure', 'animation', 'comedy', 'crime', 'documentary', 'drama', 'family', 'fantasy', 'foreign', 'history', 'horror', 'music', 'mystery', 'romance', 'sciencefiction', 'thriller', 'tvmovie', 'war', 'western']

🔑 TEMATY (Keywords): ['womandirector', 'independentfilm', 'duringcreditsstinger', 'basedonnovel', 'murder', 'aftercreditsstinger', 'violence', 'dystopia', 'sport', 'revenge', 'sex', 'friendship', 'musical', 'biography', 'teenager', '3d', 'love', 'sequel', 'suspense', 'newyork', 'police', 'losangeles', 'highschool', 'alien', 'prison', 'nudity', 'superhero', 'family', 'londonengland', 'drug', 'dyinganddeath', 'fathersonrelationship', 'daughter', 'worldwarii', 'kidnapping', 'wedding', 'remake', 'suicide', 'serialkiller', 'magic', 'friends', 'corruption', 'escape', 'basedoncomicbook', 'hospital', 'party', 'timetravel', 'basedontruestory', 'martialarts', 'airplane']

🌟 AKTORZY: ['robertdeniro', 'samuell.jackson', 'brucewillis', 'mattdamon', 'nicolascage', 'mor

In [33]:
# --- TESTY ---
moje_pomysly = [
    "animation friendship daughter family pixar",
    "horror slasher blood dark mystery",
    "action adventure tomcruise mission missionimpossible",
    "drama basedontruestory biography",
    "sciencefiction space alien jamescameron"
]

print("--- ANALIZA POMYSŁÓW ---")
for pomysl in moje_pomysly:
    wynik = predict_success(pomysl)
    print(f"Szansa: {wynik}% | Skład: {pomysl}")

# --- WŁASNY POMYSŁ ---
moj_nowy_film = "horror comedy zombieland"
print(f"\nTwój autorski pomysł ({moj_nowy_film}) -> {predict_success(moj_nowy_film)}%")

--- ANALIZA POMYSŁÓW ---
Szansa: 79.06% | Skład: animation friendship daughter family pixar
Szansa: 9.54% | Skład: horror slasher blood dark mystery
Szansa: 43.83% | Skład: action adventure tomcruise mission missionimpossible
Szansa: 50.26% | Skład: drama basedontruestory biography
Szansa: 26.07% | Skład: sciencefiction space alien jamescameron

Twój autorski pomysł (horror comedy zombieland) -> 0.37%
